In [ ]:
# IPython magig  tools
%load_ext autoreload
%autoreload 2

import json
import pandas as pd
from pathlib import Path
from aind_behavior_vr_foraging import rig

In [ ]:

base_dir = Path(r"Z:\scratch\AindBehavior.db\AindVrForaging\Rig")

rows = []

for rig_dir in base_dir.iterdir():
    if not rig_dir.is_dir():
        continue

    for json_file in rig_dir.glob("*.json"):
        with open(json_file, "r") as f:
            config = json.load(f)

        try:
            rig_model = rig.AindVrForagingRig.model_validate(config)
            channel_config = rig_model.harp_olfactometer.calibration.input.channel_config
        except:
            print(f"Error parsing {json_file}")
            continue
        
        for idx, ch in enumerate(channel_config):
            rows.append({
                "rig": rig_dir.name,
                "json_file": json_file.name,
                "channel_index": idx,
                "odorant": channel_config[ch].odorant
            })

df = pd.DataFrame(rows)

print(df)

In [ ]:
df['rig_name'] = df['json_file'].str.replace('.json', '', regex=False)
df = df.loc[df['channel_index'] != 3]
df = df.loc[~df['rig'].isin(['W10DT714679', 'TLANUSI'])]
df['specs'] = df['rig_name'].str.split('_').str[1]
df = df.loc[df.specs != 'lickometer']

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap, BoundaryNorm

pivot = df.pivot(index=["rig_name", "rig"], columns="channel_index", values="odorant")

# --- global mapping ---
all_odorants = pd.unique(pivot.values.ravel())
all_odorants = [o for o in all_odorants if pd.notna(o)]

odorant_to_int = {odor: i for i, odor in enumerate(sorted(all_odorants))}
int_to_odorant = {v: k for k, v in odorant_to_int.items()}

encoded = pivot.replace(odorant_to_int)

# --- discrete colormap ---
n = len(odorant_to_int)
cmap = ListedColormap(sns.color_palette("tab20", n))

bounds = np.arange(-0.5, n + 0.5, 1)
norm = BoundaryNorm(bounds, cmap.N)

# --- plot ---
plt.figure(figsize=(8,6))
ax = sns.heatmap(
    encoded,
    cmap=cmap,
    norm=norm,
    cbar=True
)

# fix colorbar ticks
cbar = ax.collections[0].colorbar
cbar.set_ticks(np.arange(n))
cbar.set_ticklabels([int_to_odorant[i] for i in range(n)])
ax.set_yticks(np.arange(len(encoded.index)) + 0.5)
ax.set_yticklabels(encoded.index)
plt.show()